# Construir o extender el índice en Colab (GPU)

Calcula los embeddings `BAAI/bge-m3` de los fragmentos del corpus en una GPU gratuita de
Colab (el enunciado §3.3 lo permite) y devuelve el índice listo para el proyecto.

**Incremental:** el paquete trae la caché de embeddings (`build/cache/emb/`), así que solo se
calculan los fragmentos **nuevos o modificados**. Si se agregaron 5 leyes, tarda minutos; si
es la primera vez (sin caché), ~50 min en una T4.

**Antes de abrir este cuaderno, en el computador:**
```
python -m src.corpus.build              # reprocesa el corpus (corpus_raw → fragmentos)
python -m src.index.paquete_colab       # crea dist/paquete_colab.zip (con caché)
```
Suba `dist/paquete_colab.zip` a **Google Drive → Mi unidad** (raíz).

Luego: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** → Guardar, y
ejecutar las celdas en orden (Shift + Enter).

In [ ]:
# 1. Comprobar GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Traer el paquete desde Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUTA_ZIP = "/content/drive/MyDrive/paquete_colab.zip"   # ajuste si lo guardó en otra carpeta
!rm -rf /content/proyecto && mkdir -p /content/proyecto
!unzip -q "{RUTA_ZIP}" -d /content/proyecto
%cd /content/proyecto
!ls -la build/indice build/cache/emb 2>/dev/null

## 3. Instalar dependencias (se conserva el `torch` con CUDA de Colab)

In [ ]:
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0

## 4. Construir o extender el índice en la GPU

Al final imprime cuántos embeddings **nuevos** calculó (los demás salen de la caché) y el
código de salida (0 = bien).

In [ ]:
import os
os.environ.update({"ENCODER_DEVICE": "cuda", "ENCODER_BATCH_SIZE": "64", "PYTHONIOENCODING": "utf-8",
                   "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
!python -m src.index.build; echo "CÓDIGO DE SALIDA: $? (0 = bien)"
!cat build/indice/index_manifest.json

## 5. Guardar el resultado

Crea `indice_colab.zip` (índice FAISS, BM25, manifiesto y caché actualizada), lo copia a
Drive y lo descarga. En el computador: descomprimir **en la raíz del proyecto** de modo que
queden `build/indice/index.faiss` y `build/cache/emb/…` (si Windows crea una carpeta extra
`indice_colab/`, mover su contenido `build/` a la raíz).

In [ ]:
!cd /content/proyecto && zip -q -r /content/indice_colab.zip build/indice/index.faiss build/indice/index_manifest.json build/indice/bm25 build/cache/emb
!cp /content/indice_colab.zip /content/drive/MyDrive/
!ls -la /content/indice_colab.zip
from google.colab import files
files.download("/content/indice_colab.zip")